# 강의 04 · 실습 2 — 체크포인트와 사람 개입 · (6) 고난도 III — 고객 여러 명과 꺼진 프로그램의 재개

## 1. 문제상황

- 여행사 고객센터는 고객 여러 명의 문의를 동시에 처리합니다. 고객마다 대화가 따로 이어져야 하고, 한 고객의 답장 초안이 담당자 검토를 기다리는 동안 다른 고객의 문의도 들어옵니다.
- 담당자는 퇴근 전에 검토를 끝내지 못하기도 합니다. 다음 날 프로그램을 다시 켰을 때, 어느 고객의 문의가 어느 단계에서 멈춰 있는지 프로그램이 알려 주어야 합니다.
- 검토 중이던 초안은 다시 만들지 않고 그 자리에서 이어 가야 합니다. 초안을 다시 만들면 모델 호출 비용이 다시 들고, 담당자가 이미 읽은 초안과 달라집니다.
- 반려와 다시 쓰기, 오래된 대화의 요약은 그대로 동작해야 합니다.

## 2. 문제와 목표

- **문제**: 고객마다 대화가 따로 이어져야 하는데 지금의 흐름은 고객 한 사람만 다룹니다. 프로그램이 꺼지면 어느 고객이 어느 단계에서 멈춰 있는지 알 수 없고, 다시 켜면 초안을 처음부터 다시 만들게 됩니다.
- **목표**
  - 처리 흐름(초안 → 검토 멈춤 → 승인이면 발송, 반려이면 다시 쓰기, 오래된 대화 요약)을 고객마다 다른 `thread_id`로 돌립니다.
    - 두 `thread_id`: 고객 A는 `customer-A`, 고객 B는 `customer-B`
    - 담당자의 답 둘: 승인이면 문자열 「승인」, 반려이면 사유를 담은 딕셔너리. 메일마다 새 입력을 넣을 때 반려 사유는 빈 문자열, 다시 쓴 횟수는 0으로 초기화
  - 두 고객의 문의가 모두 검토 대기 중일 때 프로그램이 꺼진 상황을 만들고, 새로 켠 프로그램이 저장 파일에서 두 `thread_id`의 멈춘 지점과 초안을 읽어 담당자에게 보여 준 뒤, 초안을 다시 만들지 않고 각 고객의 답으로 이어 가게 합니다.
  - 한 고객의 두 번째 메일에서는 오래된 대화가 요약으로 바뀝니다.
    - 사람이 정한 값: 원문으로 남기는 최근 메시지 수 2, 요약 지시문 「다음 여행사 고객센터 대화를 고객이 문의한 내용과 상담원이 안내한 처리 중심으로 두 문장 이내 한국어로 요약한다. 새 정보를 지어내지 않는다.」
    - 메일 세 통과 B의 반려 사유는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**:
  - 고객 A와 고객 B의 첫 메일이 각각 검토 노드 앞에서 멈춰 있고 두 초안이 서로 다르며,
  - 새로 만든 그래프 객체가 `get_state`로 두 `thread_id`의 멈춘 지점과 초안을 읽어 내고 그 값이 꺼지기 전과 같으며,
  - A는 승인으로 바로 발송되고, B는 반려 뒤 다시 쓴 초안이 반려 사유를 반영해 다시 멈춘 뒤 승인으로 발송되며 두 경우 모두 발송이 한 번씩이고,
  - A의 두 번째 메일에서 대화 기록의 앞부분이 요약으로 바뀌며, B의 대화 기록에는 A의 메일이 섞여 있지 않은 것을 실행 결과에서 확인합니다.
    - 출력 줄에는 「[멈춘 지점]」「[A 초안]」「[B 초안]」「[담당자에게 간 내용]」「[재개 후]」「[발송 → A]」「[발송 → B]」「[manage]」「[manage 뒤의 A 대화 기록]」「[B 대화 기록]」 표지를 붙입니다. 새로 켠 프로그램이 저장 파일에서 읽은 것은 「[A] next = (노드 이름,) / 초안 = …」 꼴로 출력합니다.
  - 담당자의 방침은 대본으로 미리 넣습니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

(「3. 워크플로우 다이어그램」에서 그린 다이어그램과 「2. 문제와 목표」의 목표를 보고 요구사항을 번호 목록으로 직접 씁니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다. 체크포인트를 저장할 파일 위치도 여기서 정합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- 체크포인트 파일은 실행할 때마다 새 임시 폴더에 만듭니다. 지난 실행의 저장 기록이 이번 실행에 섞이지 않게 하기 위해서입니다.
- 대화 기록 출력은 `show_messages(msgs)`로 합니다.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")

DB_PATH = Path(tempfile.mkdtemp(prefix="lec04_ex02_")) / "checkpoint.db"   # 체크포인트를 저장할 파일


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 앞부분 글자만 한 줄씩 출력한다."""
    for m in msgs:
        print(f"      [{type(m).__name__}] {str(m.content)[:90]}")


print("모델 준비를 마쳤습니다. 체크포인트 파일:", DB_PATH.name)


# 주어진 자료
MAIL_A1 = "출발일을 이틀 뒤로 바꾸고 싶습니다. 비용이 얼마나 드는지 알려 주세요."
MAIL_B1 = "예약한 좌석을 통로 쪽으로 바꿀 수 있나요? 추가 요금이 있는지도 궁금합니다."
MAIL_A2 = "알려 주신 대로 진행해 주세요. 변경 뒤 새 항공권은 어떻게 받나요?"
NEW_INPUT = {"feedback": "", "retries": 0, "sent": False}   # 메일마다 초기화하는 키
REJECT_REASON_B = "통로 좌석 지정 요금이 좌석 등급마다 다르다는 점을 밝힐 것."   # B의 첫 초안에 담당자가 주는 반려 사유 (대본)


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

위 실행 결과에서 다음 네 가지를 확인합니다.

1. A와 B의 첫 메일이 각각 검토 노드 앞에서 멈춰 있고(`next`에 검토 노드 이름이 출력됨), 두 초안이 서로 다른 문의에 답합니다.
2. 새 그래프 객체가 저장 파일에서 읽은 두 `thread_id`의 멈춘 지점이 꺼지기 전과 같고, 읽은 초안이 1번에서 출력한 초안과 같습니다. 모델 호출 없이 파일에서 읽은 값입니다.
3. A는 승인 뒤 발송 줄이 한 번 출력됩니다. B는 반려 뒤 발송 없이 다시 멈추고, 두 번째 초안이 반려 사유를 반영하며, 다시 쓴 횟수가 1이고, 승인 뒤 발송 줄이 한 번 출력됩니다.
4. A의 두 번째 메일에서 요약 줄이 출력되고 대화 기록에 요약 `SystemMessage`가 있습니다. 마지막에 출력한 B의 대화 기록에는 B의 메일과 답장만 있습니다.